# Análise de Dados com Python — do Pandas ao Dashboard
### Semana Acadêmica de Tecnologia da Informação e Comunicação

---

**Dataset:** [Top Spotify Songs 2023 — Kaggle](https://www.kaggle.com/datasets/nelgiriyewithana/top-spotify-songs-2023)  
**Duração:** 2h30  
**Ferramentas:** Python · Pandas · Plotly Express · Streamlit

---

### Roteiro do minicurso

| Módulo | Tema | Tempo |
|--------|------|-------|
| 01 | Ambiente, dataset e primeiros passos | 25 min |
| 02 | Pandas — limpeza e análise dos dados | 55 min |
| 03 | Visualização com Plotly Express | 30 min |
| 04 | Dashboard interativo com Streamlit | 40 min |

> 💡 **Dica:** Execute cada célula com `Shift + Enter`. Leia os comentários antes de rodar!


---
## 📦 Módulo 01 — Ambiente, dataset e primeiros passos

Vamos instalar as bibliotecas, baixar o dataset e dar o primeiro olhar nos dados.


In [ ]:
# ── Instalação das bibliotecas (necessário apenas no Colab) ──────────────────
!pip install pandas plotly streamlit -q
print("Bibliotecas instaladas!")


In [ ]:
# ── Importações ─────────────────────────────────────────────────────────────
import pandas as pd
import plotly.express as px
import warnings
warnings.filterwarnings("ignore")

print("Importações OK!")


### Carregando o dataset

Faça o download do arquivo CSV no Kaggle e envie para o Colab via painel lateral,
ou cole o link direto se o dataset estiver hospedado publicamente.

> **Link Kaggle:** https://www.kaggle.com/datasets/nelgiriyewithana/top-spotify-songs-2023


In [ ]:
# ── Carregando o dataset ─────────────────────────────────────────────────────
# Se estiver no Colab com upload manual:
df = pd.read_csv("spotify-2023.csv", encoding="latin-1")

# Se quiser usar uma URL pública (substitua pelo link correto):
# df = pd.read_csv("https://sua-url-aqui/spotify-2023.csv", encoding="latin-1")

print(f"Dataset carregado! {df.shape[0]} músicas e {df.shape[1]} colunas.")


In [ ]:
# ── Primeiras linhas ─────────────────────────────────────────────────────────
df.head()


In [ ]:
# ── Informações gerais: tipos, colunas, valores nulos ────────────────────────
df.info()


In [ ]:
# ── Estatísticas descritivas das colunas numéricas ───────────────────────────
df.describe()


> **🔍 Perguntas para a turma:**
> - Quantas músicas tem o dataset?
> - Qual coluna representa popularidade?
> - Alguma coluna tem valores faltando?


---
## Módulo 02 — Pandas: limpeza e análise dos dados
*~55 minutos*

Aqui está o coração do curso. Vamos limpar os dados e extrair insights reais do Spotify.


### 2.1 — Seleção e filtragem

In [ ]:
# ── Selecionando colunas específicas ────────────────────────────────────────
colunas_interesse = ["track_name", "artist(s)_name", "streams", "released_year",
                     "bpm", "danceability_%", "energy_%", "valence_%"]

df_sel = df[colunas_interesse].copy()
df_sel.head()


In [ ]:
# ── Filtragem: músicas lançadas após 2020 ────────────────────────────────────
df_recentes = df_sel[df_sel["released_year"] >= 2020]
print(f"Músicas lançadas em 2020 ou depois: {len(df_recentes)}")
df_recentes.head()


In [ ]:
# ── Filtragem com múltiplas condições ────────────────────────────────────────
# Músicas dançantes (danceability > 80) E energéticas (energy > 70)
df_dancantes = df_sel[
    (df_sel["danceability_%"] > 80) &
    (df_sel["energy_%"] > 70)
]
print(f"Músicas dançantes e energéticas: {len(df_dancantes)}")
df_dancantes.head()


### 2.2 — Limpeza de dados

In [ ]:
# ── Verificando valores nulos ────────────────────────────────────────────────
print("Valores nulos por coluna:")
print(df_sel.isnull().sum())


In [ ]:
# ── Corrigindo a coluna streams: converter de string para número ─────────────
# A coluna 'streams' pode vir como texto — vamos converter com segurança
df_sel["streams"] = pd.to_numeric(df_sel["streams"], errors="coerce")

# Removendo linhas com streams nulo (inválidos)
df_clean = df_sel.dropna(subset=["streams"]).copy()

print(f"Linhas antes da limpeza: {len(df_sel)}")
print(f"Linhas após a limpeza:   {len(df_clean)}")


In [ ]:
# ── Criando coluna derivada: streams em milhões ──────────────────────────────
df_clean["streams_mi"] = (df_clean["streams"] / 1_000_000).round(1)

df_clean[["track_name", "streams", "streams_mi"]].head()


### 2.3 — Agrupamentos e ranking

In [ ]:
# ── Top 10 artistas por total de streams ────────────────────────────────────
top_artistas = (
    df_clean
    .groupby("artist(s)_name")["streams_mi"]
    .sum()
    .reset_index()
    .sort_values("streams_mi", ascending=False)
    .head(10)
    .rename(columns={"artist(s)_name": "artista", "streams_mi": "streams_totais_mi"})
)

top_artistas


In [ ]:
# ── Top 10 músicas mais streamadas ──────────────────────────────────────────
top_musicas = (
    df_clean
    .nlargest(10, "streams_mi")
    [["track_name", "artist(s)_name", "streams_mi", "released_year"]]
    .rename(columns={
        "track_name": "música",
        "artist(s)_name": "artista",
        "streams_mi": "streams (mi)"
    })
    .reset_index(drop=True)
)

top_musicas


In [ ]:
# ── Pergunta de análise: músicas dançantes têm mais streams? ─────────────────
df_clean["alta_danceability"] = df_clean["danceability_%"] > 80

media_por_grupo = (
    df_clean
    .groupby("alta_danceability")["streams_mi"]
    .mean()
    .round(1)
    .reset_index()
)
media_por_grupo["alta_danceability"] = media_por_grupo["alta_danceability"].map(
    {True: "Dançabilidade alta (>80)", False: "Dançabilidade baixa (≤80)"}
)
media_por_grupo.columns = ["grupo", "média de streams (mi)"]
print(media_por_grupo.to_string(index=False))


> **🔍 Perguntas para a turma:**
> - Qual artista acumulou mais streams no dataset?
> - Músicas mais dançantes têm mais ou menos streams em média? O resultado te surpreendeu?


---
## 📊 Módulo 03 — Visualização com Plotly Express
*~30 minutos*

Vamos transformar os dados em gráficos interativos com poucas linhas de código.


### 3.1 — Gráfico de barras: top 10 artistas

In [ ]:
fig_barras = px.bar(
    top_artistas.sort_values("streams_totais_mi"),  # menor → maior para barras horizontais
    x="streams_totais_mi",
    y="artista",
    orientation="h",
    title="Top 10 artistas por total de streams (em milhões)",
    labels={"streams_totais_mi": "Streams (milhões)", "artista": ""},
    color="streams_totais_mi",
    color_continuous_scale="teal",
    text="streams_totais_mi",
)
fig_barras.update_traces(texttemplate="%{text:.0f}M", textposition="outside")
fig_barras.update_layout(
    coloraxis_showscale=False,
    plot_bgcolor="rgba(0,0,0,0)",
    height=420
)
fig_barras.show()


### 3.2 — Scatter plot: BPM × popularidade

In [ ]:
# Usando as 200 músicas mais streamadas para não poluir o gráfico
df_scatter = df_clean.nlargest(200, "streams_mi").copy()

fig_scatter = px.scatter(
    df_scatter,
    x="bpm",
    y="danceability_%",
    size="streams_mi",          # tamanho = streams
    color="energy_%",           # cor = energia
    hover_name="track_name",
    hover_data={"artist(s)_name": True, "streams_mi": True},
    title="BPM × Dançabilidade nas 200 músicas mais streamadas",
    labels={
        "bpm": "BPM (batidas por minuto)",
        "danceability_%": "Dançabilidade (%)",
        "energy_%": "Energia (%)"
    },
    color_continuous_scale="viridis",
    size_max=30,
)
fig_scatter.update_layout(plot_bgcolor="rgba(0,0,0,0)", height=480)
fig_scatter.show()


### 3.3 — Histograma: distribuição de energia

In [ ]:
fig_hist = px.histogram(
    df_clean,
    x="energy_%",
    nbins=20,
    title="Distribuição de energia nas músicas do Spotify 2023",
    labels={"energy_%": "Energia (%)", "count": "Quantidade de músicas"},
    color_discrete_sequence=["#1DB954"],  # verde Spotify
)
fig_hist.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    bargap=0.05,
    height=380
)
fig_hist.show()


> **🔍 Perguntas para a turma:**
> - No scatter, consegue identificar alguma música que você conhece passando o mouse?
> - A distribuição de energia é uniforme ou concentrada em alguma faixa?
> - Existe alguma relação visual entre BPM e dançabilidade?


---
## 🚀 Módulo 04 — Dashboard interativo com Streamlit
*~40 minutos*

Vamos transformar nossa análise em um app web real. Execute as células abaixo para gerar
o arquivo `dashboard.py`, depois rode o Streamlit no terminal.


### 4.1 — Gerando o arquivo do dashboard

A célula abaixo usa `%%writefile` para criar o `dashboard.py` diretamente do notebook.


In [ ]:
%%writefile dashboard.py
# ── dashboard.py — App Streamlit: Spotify 2023 ──────────────────────────────
import streamlit as st
import pandas as pd
import plotly.express as px

# ── Configuração da página ───────────────────────────────────────────────────
st.set_page_config(
    page_title="Spotify 2023 — Dashboard",
    page_icon="🎵",
    layout="wide"
)

# ── Carregando os dados ──────────────────────────────────────────────────────
@st.cache_data
def carregar_dados():
    df = pd.read_csv("spotify-2023.csv", encoding="latin-1")
    df["streams"] = pd.to_numeric(df["streams"], errors="coerce")
    df = df.dropna(subset=["streams"])
    df["streams_mi"] = (df["streams"] / 1_000_000).round(1)
    return df

df = carregar_dados()

# ── Cabeçalho ────────────────────────────────────────────────────────────────
st.title("🎵 Spotify 2023 — Análise de dados")
st.caption("Minicurso de Análise de Dados com Python · SEATIC")

st.divider()

# ── Filtro lateral ───────────────────────────────────────────────────────────
st.sidebar.header("Filtros")

anos_disponiveis = sorted(df["released_year"].dropna().unique().tolist(), reverse=True)
ano_selecionado = st.sidebar.selectbox("Ano de lançamento", ["Todos"] + anos_disponiveis)

if ano_selecionado != "Todos":
    df_filtrado = df[df["released_year"] == ano_selecionado]
else:
    df_filtrado = df.copy()

# ── Métricas de destaque ─────────────────────────────────────────────────────
col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric("Total de músicas", f"{len(df_filtrado):,}")

with col2:
    top_artista = (
        df_filtrado.groupby("artist(s)_name")["streams_mi"].sum().idxmax()
    )
    st.metric("Artista #1 em streams", top_artista)

with col3:
    musica_top = df_filtrado.nlargest(1, "streams_mi").iloc[0]
    st.metric("Música mais streamada", musica_top["track_name"])

with col4:
    mais_energetica = df_filtrado.nlargest(1, "energy_%").iloc[0]
    st.metric("Mais energética", mais_energetica["track_name"])

st.divider()

# ── Gráficos ─────────────────────────────────────────────────────────────────
col_a, col_b = st.columns(2)

with col_a:
    st.subheader("Top 10 artistas por streams")
    top_art = (
        df_filtrado.groupby("artist(s)_name")["streams_mi"]
        .sum()
        .reset_index()
        .sort_values("streams_mi", ascending=False)
        .head(10)
        .sort_values("streams_mi")
    )
    fig_bar = px.bar(
        top_art,
        x="streams_mi", y="artist(s)_name",
        orientation="h",
        labels={"streams_mi": "Streams (mi)", "artist(s)_name": ""},
        color="streams_mi",
        color_continuous_scale="teal",
    )
    fig_bar.update_layout(coloraxis_showscale=False, height=380,
                          plot_bgcolor="rgba(0,0,0,0)")
    st.plotly_chart(fig_bar, use_container_width=True)

with col_b:
    st.subheader("Distribuição de energia")
    fig_hist = px.histogram(
        df_filtrado, x="energy_%", nbins=20,
        labels={"energy_%": "Energia (%)", "count": "Músicas"},
        color_discrete_sequence=["#1DB954"],
    )
    fig_hist.update_layout(bargap=0.05, height=380,
                           plot_bgcolor="rgba(0,0,0,0)")
    st.plotly_chart(fig_hist, use_container_width=True)

st.subheader("BPM × Dançabilidade (top 200 músicas)")
df_scatter = df_filtrado.nlargest(200, "streams_mi")
fig_sc = px.scatter(
    df_scatter,
    x="bpm", y="danceability_%",
    size="streams_mi", color="energy_%",
    hover_name="track_name",
    hover_data={"artist(s)_name": True, "streams_mi": True},
    labels={"bpm": "BPM", "danceability_%": "Dançabilidade (%)", "energy_%": "Energia (%)"},
    color_continuous_scale="viridis",
    size_max=28,
)
fig_sc.update_layout(plot_bgcolor="rgba(0,0,0,0)", height=420)
st.plotly_chart(fig_sc, use_container_width=True)

# ── Tabela interativa ────────────────────────────────────────────────────────
st.divider()
with st.expander("Ver tabela completa dos dados filtrados"):
    st.dataframe(
        df_filtrado[["track_name", "artist(s)_name", "released_year",
                     "streams_mi", "bpm", "danceability_%", "energy_%"]]
        .sort_values("streams_mi", ascending=False)
        .rename(columns={
            "track_name": "Música",
            "artist(s)_name": "Artista",
            "released_year": "Ano",
            "streams_mi": "Streams (mi)",
            "bpm": "BPM",
            "danceability_%": "Dançabilidade (%)",
            "energy_%": "Energia (%)"
        })
        .reset_index(drop=True),
        use_container_width=True,
        height=300
    )

st.caption("Fonte: Kaggle — Top Spotify Songs 2023")


### 4.2 — Rodando o dashboard

Após executar a célula acima, rode o comando abaixo no terminal (ou em uma nova célula no Colab):


In [ ]:
# ── No Colab, use este comando para rodar o Streamlit ────────────────────────
# Instala o localtunnel para expor o app publicamente durante a aula
!pip install streamlit pyngrok -q

from pyngrok import ngrok
import subprocess, threading, time

def run_streamlit():
    subprocess.run(["streamlit", "run", "dashboard.py",
                    "--server.port", "8501",
                    "--server.headless", "true"])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

time.sleep(4)  # aguarda o servidor subir

public_url = ngrok.connect(8501)
print(f"\n🚀 Dashboard disponível em: {public_url}")
print("   Abra o link acima no navegador para ver o app ao vivo!")


### 4.3 — Executando localmente (fora do Colab)

Se estiver no computador local, use o terminal:

```bash
# Instalar dependências
pip install pandas plotly streamlit

# Rodar o dashboard
streamlit run dashboard.py
```

O Streamlit abrirá automaticamente no navegador em `http://localhost:8501`.


---
## 🏁 Encerramento e próximos passos

Parabéns! Você passou por toda a pipeline de análise de dados:

```
CSV bruto → Pandas (limpeza + análise) → Plotly (visualização) → Streamlit (dashboard)
```

### 🚀 Desafio pós-evento

Publique seu dashboard gratuitamente no **Streamlit Community Cloud**:

1. Suba o `dashboard.py` e o `spotify-2023.csv` em um repositório público no GitHub
2. Acesse [share.streamlit.io](https://share.streamlit.io) e conecte o repositório
3. Clique em **Deploy** — em menos de 2 minutos seu app estará online!
4. Envie o link para a instrutora 🎯

### 📚 O que aprender a seguir

| Tema | Ferramenta | Por quê |
|------|-----------|---------|
| Banco de dados | SQLite + Pandas | Substituir CSVs por dados persistentes |
| Mais visualizações | Seaborn, Matplotlib | Gráficos para publicação acadêmica |
| Machine Learning | scikit-learn | Prever popularidade de músicas |
| APIs | requests + FastAPI | Buscar dados do Spotify ao vivo |
| Deploy avançado | Docker + Railway | Colocar apps em produção |

---
*Minicurso preparado para a Semana Acadêmica de TIC* 🎓
